In [1]:
print("Python")

Python


In [2]:
import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
words = open("names.txt",'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [4]:
# building the vocab of characters and mappings to/from intergers
chars =  sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos = {i:s for s,i in stoi.items()}
vocab_size=len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [5]:
# build the dataset 
block_size=3 # context length: how many characters do we take to predict the next one?
def build_dataset(words):
    X,Y = [],[]
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix=stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X=torch.tensor(X)
    Y=torch.tensor(Y)
    print(X.shape,Y.shape)
    return X,Y

import random 
random.seed(42)
random.shuffle(words)
n1=int(0.8* len(words))
n2= int(0.9* len(words))

Xtr,Ytr= build_dataset(words[:n1])      #80%
Xdev,Ydev = build_dataset(words[n1:n2]) #10%
Xte, Yte=build_dataset(words[n2:])      #10%


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [6]:
#utility function we will use later when comparing manual gradients to Pytorch gradients

def cmp(s,dt, t):
    ex=torch.all(dt==t.grad).item()
    app = torch.allclose(dt,t.grad)
    maxdiff = (dt - t.grad).abs().max().item() 
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')
    

In [7]:
n_embd=10
n_hidden = 64
g=torch.Generator().manual_seed(2147483647) # for reproduucibility
C = torch.randn((vocab_size, n_embd),                   generator=g)
#  Layer 1
W1= torch.randn((n_embd * block_size, n_hidden),        generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1= torch.randn(n_hidden,                               generator=g) * 0.1
# Layer 2
W2= torch.randn((n_hidden, vocab_size),                 generator=g) * 0.1
b2= torch.randn(vocab_size,                             generator=g) * 0.1
# BatchNorm parameters 
bngain = torch.randn((1, n_hidden)) *0.1 + 1.0
bnbias = torch.randn((1, n_hidden)) * 0.1

parameters=[C,W1,b1,W2,b2, bngain,bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requres_grad = True

4137


In [ ]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibacth
ix = torch.randint(0,Xtr.shape[0], (batch_size,), generator=g)
Xb,Yb = Xtr[ix], Ytr[ix] # bacth X, Y

In [ ]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time
emb= C[Xb] # embeded the characters into vectors 
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat@ W1 +b1
# BatchNorm Layer 
bnmeani = 1/n*hprebn.sum(0,keepdim=True)
bndiff=hprebn - bnmeani
bndiff2=bndiff**2
bnvar=1/(n-1)*(bndiff2).sum(0,keepdim=True) # note: Bessel's Correction (dividing by n-1, not n)
bnvar_inv= (bnvar+ 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact =  bngain * bnraw + bnbias
# Non-Linearity 
h= torch.tanh(hpreact) #hidden layer
#Linear Layer 2
logits = h@ W2 + b2 # output layer

#corss entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes =  logits.max(1,keepdim=True).values
norm_logits = logits - logit_maxes #subtract max for numerical stability 
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum **-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs =  counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

#PyTorch backward pass 
for p in parameters:
    p.grad = None
for t in [logprobs, probs , counts , counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits , h, hpreact, bnraw,
          bnvar_inv,bnvar, bndiff2, bndiff, hprebn,bnmeani,
          embcat, emb]:
    t.retain_grad()
loss.backward()
loss

RuntimeError: can't retain_grad on Tensor that has requires_grad=False

In [ ]:
# backpropagating through exactly all of the variables
# as they are defined in the foward pass above, one by one

# cmp('logprobs', dlogprobs, logprobs)
# cmp('probs',dprobs,probs)
# cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
# cmp('counts_sum', dcounts_sum, counts_sum)
# cmp('counts', dcounts, counts)
# cmp('norm_logits', dnorm_logits, norm_logits)
# cmp('logit_maxes', dlogit_maxes, logit_axes)
# cmp('logits', dlogits, logits)
# cmp('h',dh, h)
# cmp("w2", dW2, W2)
# cmp('b2', db2, b2)
# cmp('hpreact', dhpreact, hpreact)
# cmp('bngain', dbngain, bngain)
# cmp('bnbias',dbnbias, bnbias)
# cmp('bnraw', dbnraw, bnraw)
# cmp('bnrav_inv',dbnvar_inv, bnvar_inv)
# cmp('bnvar',dbnvar, bnvar)
# cmp('bndiff2', dbndiff2, bndiff2)
# cmp('bndiff', dbndiff, bndiff)
# cmp('bnmeani', dbmneani,bnmeani)
# cmp('hprebn', bhprebn, hprebn)
# cmp('embcat', dembcat, embcat)
# cmp('W1', dW1, W1)
# cmp('emb', demb, emb)
# cmp('C', dC,C)

In [1]:
# foward pass
# before:
# logit_maxes = logits.max(1,keepdim=True).values
# norm_logits=logits-logit_maxes # subract  max for numerical stability 
# counts=norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum **-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs =  counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits,Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

NameError: name 'F' is not defined

In [2]:
# backward_pass


# cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

In [ ]:
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t its input , simply the expression, and just write it out 

# foward pass 

# before:
# bnmeani = 1/n*hprebn.sum(0,keepdim=True)
# bndiff=hprebn - bnmeani
# bndiff2=bndiff**2
# bnvar=1/(n-1)*(bndiff2).sum(0,keepdim=True) # note: Bessel's Correction (dividing by n-1, not n)
# bnvar_inv= (bnvar+ 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact =  bngain * bnraw + bnbias

# now:
# batch_norm uses biased variance so sadly just calling this doesn't work
# hpreact2 = F.batch_norm(hprebn, None , None ,bngain,bnbias, training=True)
# sadly unresolved, apparently : https://github.com/pytorch/pytorch/pytorch/issues/19902
# so instead implementing the correct foward pass here explicitly but one line:
hpreact_fast = bngain * (hprebn-hprebn.mean(0,keepdim=True)) / torch.sqrt(hprebn.var(0,keepdim=True,unbiased=True))
print(hpreact_fast , 'max_diff:', (hpreact_fast - hpreact).abs().max())


In [ ]:
# cmp('hprebn', dhprebn,hprebn) # I can only get approximate to be true, my maxdiff is 9e-10

In [ ]:
# Train the MLP neural nnet with your own backwards pass


# init
n_embd=10 # the dimensionality of the character embedding vectors 
n_hidden=200 # the number of neurons in the hidden layer of the MLP

g=torch.Generator().manual_seed(2147483647) # for reproduucibility
C = torch.randn((vocab_size, n_embd),                   generator=g)
#  Layer 1
W1= torch.randn((n_embd * block_size, n_hidden),        generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1= torch.randn(n_hidden,                               generator=g) * 0.1
# Layer 2
W2= torch.randn((n_hidden, vocab_size),                 generator=g) * 0.1
b2= torch.randn(vocab_size,                             generator=g) * 0.1
# BatchNorm parameters 
bngain = torch.randn((1, n_hidden)) *0.1 + 1.0
bnbias = torch.randn((1, n_hidden)) * 0.1

parameters=[C,W1,b1,W2,b2, bngain,bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requres_grad = True

# same optimization as last time
max_steps=200000
batch_size = 32
n = batch_size # convenience
lossi = []

# use this context manager for efficiency once your backward pass is written  (TODO)
# with torch.no_grad():

# Kick off optimization 
for i in range(max_steps):
    
    # minibatch construct 
    ix = torch.randint(0,Xtr.shape[0], (batch_size,), generator=g)
    Xb,Yb = Xtr[ix], Ytr[ix] # bacth X, Y

    # foward pass
    emb =  C[Xb]
    embcat =  emb.view(emb.shape[0],-1) # concatenate the vectors 

    # Linear layer
    hprebn = embcat @ W1 + b1 # hidden layer pre-activation
    # BatchNorm layer

    bnmean = hprebn.mean(0,keepdim=True)
    bnvar=hprebn.var(0,keepdim=True, unbiased=True)
    bnvar_inv= (bnvar+1e-5)**-0.5
    bnraw=(hprebn-bnmean) * bnvar_inv
    hpreact =  bngain * bnraw + bnbias 

    # Non Linearlity
    h = torch.tanh(hpreact) # hidden layer
    logits = h @ W2 + b2 #output layer
    loss = F.cross_entropy(logits,Yb) # Loss function

    # backward pass 
    for p in parameters:
        p.grad=None
    loss.backward() # use this for correctness comparisons, delete it later !

    # Manual backprop! #swole_doge_meme
    # 
    dC,dW1,db1,dW2,db2,dbngain,dbnbias= None, None,None,None,None,None,None
    grads= [dC,dW1,db1,dW2,db2,dbngain,dbnbias]

    # update
    lr = 0.1 if i<100000 else 0.01 # step learning rate decay 
    for p, grad in zip(parameters,grads):
        p.data += -lr*p.grad   #old way of cheems doge (using  PyTorch grad from .backward())
        # p.data += -lr * grad # new way of swole doge TODO: enable

    # track stats
    if i% 10000==0:#print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

    if i >= 100:
        break

In [3]:
# useful for checking your gradients
# for p,g in zip(parameters,grads):
#     cmp(str(tuple(p.shape)), g,p)


In [6]:
# # calibrate the batch norm at the end of training

# with torch.no_grad():
#     # pass the training set through
#     emb=C[Xtr]
#     embcat = emb.view(emb.shape[0],-1)
#     hpreact=embcat @ W1+b1
#     # measure the mean/std over the entire training set 
#     bnmean = hpreact.mean(0, keepdim=True)
#     bnvar=hpreact.var(0, keepdim=True, unbiased=True)


In [ ]:
# evaluate train and val loss
torch.no_grad() # this decorator disables gradient tracking 
def split_loss(split):
    x,y={
        'train':(Xtr,Ytr),
        'val': (Xdev,Ydev),
        'test': (Xte,Yte),
    }[split]
    emb=C[x] # (N,block_size, n_embd)
    embcat=emb.view(emb.shape[0], -1) # concat into (N, block_size* n_embd)
    hpreact = embcat @ W1 + b1
    hpreact =bngain * (hpreact-bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
    h=torch.tanh(hpreact ) # (N,n_hidden)
    logits = h @ W2 + b2 # (N,vocab_size)
    loss = F.cross_entropy(logits,y)
    print(split,loss.item())

split_loss("train")
split_loss('val')

In [ ]:
# I achieved:
# train 
# val 